# Basic RAG Starter

A minimal RAG baseline using minsearch, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

Replace the example data with your own and adapt the instructions to match what your project should actually do.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Replace this example with your own data. Each document should be a dict with a few text fields you want to search over.

In [ ]:
import os
import duckdb

# ── 1. Create a synthetic star-schema DuckDB database ─────────────────────────
db_path = "../data/sales.duckdb"
os.makedirs("../data", exist_ok=True)

con = duckdb.connect(db_path)
con.execute("DROP TABLE IF EXISTS orders")
con.execute("DROP TABLE IF EXISTS products")
con.execute("DROP TABLE IF EXISTS customers")

con.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        VARCHAR,
    segment     VARCHAR,   -- Enterprise | Mid-Market | SMB
    country     VARCHAR
)""")

con.execute("""
CREATE TABLE products (
    product_id  INTEGER PRIMARY KEY,
    name        VARCHAR,
    category    VARCHAR,   -- Software | Hardware | Services
    unit_price  DOUBLE
)""")

con.execute("""
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers(customer_id),
    product_id  INTEGER REFERENCES products(product_id),
    order_date  DATE,
    quantity    INTEGER,
    region      VARCHAR    -- North | South | East | West
)""")

con.execute("""
INSERT INTO customers VALUES
  (1,'Acme Corp','Enterprise','US'),(2,'Globex','Mid-Market','UK'),
  (3,'Initech','SMB','DE'),(4,'Umbrella','Enterprise','US'),
  (5,'Hooli','Mid-Market','US'),(6,'Dunder Mifflin','SMB','US'),
  (7,'Pied Piper','SMB','US'),(8,'Massive Dynamics','Enterprise','UK'),
  (9,'Cyberdyne','Enterprise','US'),(10,'Soylent Corp','Mid-Market','CA')
""")

con.execute("""
INSERT INTO products VALUES
  (1,'Analytics Suite','Software',1200.00),(2,'Data Connector','Software',450.00),
  (3,'Server Rack','Hardware',3500.00),(4,'Laptop Pro','Hardware',1800.00),
  (5,'Consulting Day','Services',2000.00),(6,'Training Pack','Services',800.00),
  (7,'Cloud Gateway','Software',600.00),(8,'NAS Device','Hardware',1100.00)
""")

con.execute("""
INSERT INTO orders VALUES
  (1,1,1,'2024-01-15',3,'North'),(2,1,5,'2024-02-10',2,'North'),
  (3,2,2,'2024-01-20',5,'East'), (4,2,4,'2024-03-05',1,'East'),
  (5,3,6,'2024-02-28',2,'South'),(6,4,1,'2024-01-30',10,'West'),
  (7,4,3,'2024-04-12',2,'West'), (8,5,7,'2024-03-18',4,'North'),
  (9,6,6,'2024-04-01',3,'South'),(10,7,2,'2024-04-22',6,'East'),
  (11,8,1,'2024-05-03',5,'East'), (12,8,5,'2024-05-10',3,'East'),
  (13,9,3,'2024-02-14',1,'West'), (14,9,4,'2024-06-01',4,'West'),
  (15,10,7,'2024-06-15',8,'North'),(16,1,8,'2024-06-20',2,'North'),
  (17,3,1,'2024-07-04',1,'South'),(18,5,5,'2024-07-11',2,'North'),
  (19,6,4,'2024-08-03',3,'South'),(20,2,8,'2024-08-19',4,'East')
""")

con.close()
print("Database created at", db_path)

# ── 2. Define the RAG corpus: schema descriptions + example NL→SQL pairs ──────
documents = [
    # --- Schema entries ---
    {
        "type": "schema",
        "table_name": "customers",
        "description": (
            "customers(customer_id INT PK, name VARCHAR, segment VARCHAR, country VARCHAR). "
            "One row per customer. segment is one of: Enterprise, Mid-Market, SMB."
        ),
        "question": "",
        "sql": "",
    },
    {
        "type": "schema",
        "table_name": "products",
        "description": (
            "products(product_id INT PK, name VARCHAR, category VARCHAR, unit_price DOUBLE). "
            "One row per product. category is one of: Software, Hardware, Services."
        ),
        "question": "",
        "sql": "",
    },
    {
        "type": "schema",
        "table_name": "orders",
        "description": (
            "orders(order_id INT PK, customer_id INT FK→customers, product_id INT FK→products, "
            "order_date DATE, quantity INT, region VARCHAR). "
            "One row per order line. region is one of: North, South, East, West. "
            "Revenue = quantity * unit_price (join products)."
        ),
        "question": "",
        "sql": "",
    },
    # --- Example NL→SQL pairs ---
    {
        "type": "example",
        "table_name": "orders",
        "description": "Total revenue per region, sorted highest first",
        "question": "What is the total revenue by region?",
        "sql": (
            "SELECT region, ROUND(SUM(o.quantity * p.unit_price), 2) AS total_revenue "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY region ORDER BY total_revenue DESC"
        ),
    },
    {
        "type": "example",
        "table_name": "customers",
        "description": "Top customers ranked by total spend",
        "question": "Who are the top 5 customers by total revenue?",
        "sql": (
            "SELECT c.name, ROUND(SUM(o.quantity * p.unit_price), 2) AS total_spend "
            "FROM orders o "
            "JOIN customers c USING (customer_id) "
            "JOIN products p USING (product_id) "
            "GROUP BY c.name ORDER BY total_spend DESC LIMIT 5"
        ),
    },
    {
        "type": "example",
        "table_name": "orders",
        "description": "Monthly revenue trend across all regions",
        "question": "What is the monthly revenue trend?",
        "sql": (
            "SELECT strftime(order_date, '%Y-%m') AS month, "
            "ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY month ORDER BY month"
        ),
    },
    {
        "type": "example",
        "table_name": "products",
        "description": "Revenue breakdown by product category",
        "question": "Which product category generates the most revenue?",
        "sql": (
            "SELECT p.category, ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY p.category ORDER BY revenue DESC"
        ),
    },
    {
        "type": "example",
        "table_name": "customers",
        "description": "Revenue by customer segment (Enterprise, Mid-Market, SMB)",
        "question": "How does revenue compare across customer segments?",
        "sql": (
            "SELECT c.segment, ROUND(SUM(o.quantity * p.unit_price), 2) AS revenue "
            "FROM orders o "
            "JOIN customers c USING (customer_id) "
            "JOIN products p USING (product_id) "
            "GROUP BY c.segment ORDER BY revenue DESC"
        ),
    },
    {
        "type": "example",
        "table_name": "orders",
        "description": "Number of orders and total quantity sold per product",
        "question": "How many orders does each product have?",
        "sql": (
            "SELECT p.name, COUNT(*) AS num_orders, SUM(o.quantity) AS total_qty "
            "FROM orders o JOIN products p USING (product_id) "
            "GROUP BY p.name ORDER BY num_orders DESC"
        ),
    },
]

print(f"Loaded {len(documents)} documents into corpus "
      f"({sum(1 for d in documents if d['type']=='schema')} schema, "
      f"{sum(1 for d in documents if d['type']=='example')} examples)")


## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [ ]:
index = Index(
    text_fields=["description", "question", "sql"],
    keyword_fields=["type", "table_name"],
)

index.fit(documents)
print("Index ready —", len(documents), "documents indexed")


## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=5)

In [ ]:
instructions = """
You are a DuckDB SQL expert. Your job is to turn a business user's natural-language
question into a single, valid DuckDB SQL query.

Rules:
- Use ONLY the tables and columns described in the SCHEMA entries in CONTEXT.
- Prefer the patterns shown in the EXAMPLE entries in CONTEXT when they are relevant.
- Revenue is always quantity * unit_price (requires joining orders with products).
- Return ONLY a JSON object with two keys, no markdown fences:
    {"sql": "<the SQL query>", "explanation": "<one sentence plain-English explanation>"}
""".strip()


def build_prompt(query, search_results):
    schema_entries = [r for r in search_results if r.get("type") == "schema"]
    example_entries = [r for r in search_results if r.get("type") == "example"]

    schema_block = "\n".join(
        f"- {r['description']}" for r in schema_entries
    ) or "(no schema retrieved)"

    example_block = "\n".join(
        f"- Q: {r['question']}\n  SQL: {r['sql']}"
        for r in example_entries
        if r.get("question")
    ) or "(no examples retrieved)"

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
SCHEMA:
{schema_block}

EXAMPLES:
{example_block}
</CONTEXT>
""".strip()

    return user_prompt


In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    raw = llm(prompt, instructions)

    # Parse the JSON the LLM returns
    try:
        parsed = json.loads(raw)
        sql = parsed.get("sql", "").strip()
        explanation = parsed.get("explanation", "")
    except json.JSONDecodeError:
        # Fallback: treat raw response as SQL if it isn't valid JSON
        sql = raw.strip()
        explanation = ""

    # Execute SQL against DuckDB
    con = duckdb.connect("../data/sales.duckdb", read_only=True)
    try:
        results = con.execute(sql).df().to_dict(orient="records")
        error = None
    except Exception as e:
        results = []
        error = str(e)
    finally:
        con.close()

    return {
        "query":       query,
        "sql":         sql,
        "explanation": explanation,
        "results":     results,
        "error":       error,
        "trust": {
            "schema_docs_used":  [r["table_name"] for r in search_results if r.get("type") == "schema"],
            "examples_retrieved": sum(1 for r in search_results if r.get("type") == "example"),
        },
    }


## 4. Try It Out

In [ ]:
result = rag("Which region had the highest revenue, and how does each region compare?")

print("Query      :", result["query"])
print("SQL        :", result["sql"])
print("Explanation:", result["explanation"])
print("Trust      :", result["trust"])
print("Error      :", result["error"])
print("\nResults:")
for row in result["results"]:
    print(" ", row)


Use `uv add PACKAGE_NAME` to add a new dependency to pyproject.toml.
